
Continuous trial

In [4]:
import numpy as np
import pandas as pd

def generate_continuous_trial(
    n_per_arm,
    baseline_mean=50.0,
    baseline_sd=10.0,
    baseline_outcome_corr=0.5,
    outcome_sd=10.0,
    treatment_effect=5.0,
    dropout_rate=0.0,
    allocation_ratio=1.0,
    seed=None,
):

    rng = np.random.default_rng(seed)

    n_treat = int(round(n_per_arm * allocation_ratio))
    n_total = n_per_arm + n_treat

    arm = np.array([0] * n_per_arm + [1] * n_treat)
    baseline = rng.normal(baseline_mean, baseline_sd, size=n_total)

    beta_baseline = baseline_outcome_corr * (outcome_sd / baseline_sd)
    residual_sd = outcome_sd * np.sqrt(1 - baseline_outcome_corr**2)

    outcome_mean = (
        baseline_mean
        + beta_baseline * (baseline - baseline_mean)
        + treatment_effect * arm
    )
    outcome = outcome_mean + rng.normal(0, residual_sd, size=n_total)

    if dropout_rate > 0:
        dropout_mask = rng.random(n_total) < dropout_rate
        outcome = np.where(dropout_mask, np.nan, outcome)

    return pd.DataFrame({
        "subject_id": np.arange(1, n_total + 1),
        "arm": arm,
        "baseline": baseline,
        "outcome": outcome,
    })

In [2]:
check = generate_continuous_trial(n_per_arm=200_000, treatment_effect=5.0, dropout_rate=0.15, seed=1)
print(check.groupby("arm")["outcome"].mean().diff().iloc[-1])  # should land near 5.0
print(check["outcome"].isna().mean())  # should land near 0.15

4.976518862133574
0.1496425


Binary trial

In [2]:
def generate_binary_trial(
    n_per_arm,
    control_event_rate=0.30,
    odds_ratio=0.5,
    baseline_mean=50.0,
    baseline_sd=10.0,
    baseline_log_odds_coef=0.0,
    dropout_rate=0.0,
    allocation_ratio=1.0,
    seed=None,
):
    
    rng = np.random.default_rng(seed)

    n_treat = int(round(n_per_arm * allocation_ratio))
    n_total = n_per_arm + n_treat

    arm = np.array([0] * n_per_arm + [1] * n_treat)
    baseline = rng.normal(baseline_mean, baseline_sd, size=n_total)

    intercept = np.log(control_event_rate / (1 - control_event_rate))
    beta_treatment = np.log(odds_ratio)

    logit_p = (
        intercept
        + beta_treatment * arm
        + baseline_log_odds_coef * (baseline - baseline_mean)
    )
    p = 1 / (1 + np.exp(-logit_p))
    event = rng.binomial(1, p).astype(float)

    if dropout_rate > 0:
        dropout_mask = rng.random(n_total) < dropout_rate
        event = np.where(dropout_mask, np.nan, event)

    return pd.DataFrame({
        "subject_id": np.arange(1, n_total + 1),
        "arm": arm,
        "baseline": baseline,
        "event": event,
    })

In [ ]:
check = generate_binary_trial(n_per_arm=200_000, control_event_rate=0.30, odds_ratio=0.5, dropout_rate=0.10, seed=1)
print(check.groupby("arm")["event"].mean())  # control ~0.30, treatment should be lower given OR 0.5
print(check["event"].isna().mean())  # should land near 0.10

arm
0    0.300361
1    0.176201
Name: event, dtype: float64
0.1002025
        subject_id  arm   baseline  event
0                1    0  53.455842    1.0
1                2    0  58.216181    NaN
2                3    0  53.304371    0.0
3                4    0  36.968428    1.0
4                5    0  59.053559    1.0
...            ...  ...        ...    ...
399995      399996    1  36.288715    0.0
399996      399997    1  47.403685    1.0
399997      399998    1  57.945830    1.0
399998      399999    1  52.992131    0.0
399999      400000    1  64.171409    0.0

[400000 rows x 4 columns]


Survival trial

In [6]:
def generate_survival_trial(
    n_per_arm,
    control_median_survival=12.0,
    hazard_ratio=0.6,
    baseline_mean=50.0,
    baseline_sd=10.0,
    baseline_log_hr_coef=0.0,
    accrual_period=6.0,
    follow_up_period=18.0,
    dropout_rate=0.0,
    allocation_ratio=1.0,
    seed=None,
):
    
    rng = np.random.default_rng(seed)

    n_treat = int(round(n_per_arm * allocation_ratio))
    n_total = n_per_arm + n_treat

    arm = np.array([0] * n_per_arm + [1] * n_treat)
    baseline = rng.normal(baseline_mean, baseline_sd, size=n_total)

    h0 = np.log(2) / control_median_survival
    linear_predictor = (
        np.log(hazard_ratio) * arm
        + baseline_log_hr_coef * (baseline - baseline_mean)
    )
    hazard = h0 * np.exp(linear_predictor)
    true_event_time = rng.exponential(1 / hazard)

    enrollment_time = rng.uniform(0, accrual_period, size=n_total)
    study_end = accrual_period + follow_up_period
    admin_censor_time = np.clip(study_end - enrollment_time, 0, None)

    if dropout_rate > 0:
        dropout_hazard = -np.log(1 - dropout_rate) / follow_up_period
        dropout_time = rng.exponential(1 / dropout_hazard, size=n_total)
    else:
        dropout_time = np.full(n_total, np.inf)

    observed_time = np.minimum.reduce([true_event_time, admin_censor_time, dropout_time])
    event = (observed_time == true_event_time).astype(int)

    return pd.DataFrame({
        "subject_id": np.arange(1, n_total + 1),
        "arm": arm,
        "baseline": baseline,
        "time": observed_time,
        "event": event,
    })

In [7]:
check = generate_survival_trial(n_per_arm=200_000, control_median_survival=12.0, hazard_ratio=0.6, dropout_rate=0.05, seed=1)
summary = check.groupby("arm").apply(
    lambda d: pd.Series({"events": d["event"].sum(), "person_time": d["time"].sum()}),
    include_groups=False,
)
summary["incidence_rate"] = summary["events"] / summary["person_time"]
irr = summary["incidence_rate"].iloc[1] / summary["incidence_rate"].iloc[0]
print(summary)
print(irr)  

       events   person_time  incidence_rate
arm                                        
0    137121.0  2.368324e+06        0.057898
1    100435.0  2.902597e+06        0.034602
0.5976342797170259
